<a href="https://colab.research.google.com/github/arthur-craveiro1/calculo-numerico/blob/master/Atividade03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade de CN - 03

## Exercício 01

In [6]:
def bissecao(f, a, b, tol=1e-6, max_iter=100):
    aproximacoes = []

    # A bisseção exige mudança de sinal no intervalo
    if f(a) * f(b) > 0:
        return None, 0, aproximacoes, False

    x_anterior = None

    for k in range(1, max_iter + 1):
        x = (a + b) / 2
        aproximacoes.append(x)

        # Critérios de parada
        if abs(f(x)) < tol:
            return x, k, aproximacoes, True

        if x_anterior is not None:
            if abs(x - x_anterior) < tol:
                return x, k, aproximacoes, True

        # Escolhe o novo intervalo
        if f(a) * f(x) < 0:
            b = x
        else:
            a = x

        x_anterior = x

    return x, max_iter, aproximacoes, False

def newton(f, df, x0, tol=1e-6, max_iter=100):
    aproximacoes = [x0]
    x = x0

    for k in range(1, max_iter + 1):

        # Evita divisão por zero
        if abs(df(x)) < 1e-12:
            return x, k - 1, aproximacoes, False

        x_novo = x - f(x) / df(x)
        aproximacoes.append(x_novo)

        # Critérios de parada
        if abs(x_novo - x) < tol or abs(f(x_novo)) < tol:
            return x_novo, k, aproximacoes, True

        x = x_novo

    return x, max_iter, aproximacoes, False


def secante(f, x0, x1, tol=1e-6, max_iter=100):
    aproximacoes = [x0, x1]

    for k in range(1, max_iter + 1):
        denominador = f(x1) - f(x0)

        # Evita divisão por zero
        if abs(denominador) < 1e-12:
            return x1, k - 1, aproximacoes, False

        x2 = x1 - f(x1) * (x1 - x0) / denominador
        aproximacoes.append(x2)

        # Critérios de parada
        if abs(x2 - x1) < tol or abs(f(x2)) < tol:
            return x2, k, aproximacoes, True

        x0 = x1
        x1 = x2

    return x1, max_iter, aproximacoes, False

## Critério de parada

Foi adotada uma tolerância de

$$
\text{tol} = 10^{-6}
$$

e um limite máximo de **100 iterações**.

Para determinar a convergência dos métodos, foram utilizados dois critérios de parada. O primeiro verifica a diferença entre duas aproximações consecutivas:

$$
|x_{k+1} - x_k| < 10^{-6}
$$

O segundo verifica o resíduo da função na aproximação obtida:

$$
|f(x_{k+1})| < 10^{-6}
$$

O método é considerado convergente quando pelo menos um desses critérios é satisfeito.

Além disso, foram consideradas situações em que não é possível continuar o processo iterativo:

- **Bisseção:** o intervalo inicial deve apresentar mudança de sinal, isto é,

$$
f(a)f(b) \leq 0
$$

Caso

$$
f(a)f(b) > 0
$$

não há garantia de uma raiz no intervalo e o método é interrompido.

- **Newton:** o método utiliza a expressão

$$
x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)}
$$

Portanto, se a derivada $f'(x_k)$ for igual ou muito próxima de zero, a iteração é interrompida para evitar uma divisão por zero.

- **Secante:** o método utiliza

$$
x_{k+1}
=
x_k
-
f(x_k)
\frac{x_k-x_{k-1}}
{f(x_k)-f(x_{k-1})}
$$

Assim, se

$$
f(x_k)-f(x_{k-1})
$$

for igual ou muito próximo de zero, o método é interrompido.

Caso nenhum dos critérios de convergência seja satisfeito após **100 iterações**, o método é considerado como não convergente para as condições iniciais utilizadas.

## Exercício 02

In [7]:
import numpy as np
import pandas as pd

# Funções
def f1(x):
    return x**2 - 2

def df1(x):
    return 2*x


def f2(x):
    return x**3 - x - 2

def df2(x):
    return 3*x**2 - 1


def f3(x):
    return np.exp(-x) - x

def df3(x):
    return -np.exp(-x) - 1


def f4(x):
    return x**3 - 2*x + 2

def df4(x):
    return 3*x**2 - 2


# Dados de cada função:
# nome, função, derivada, intervalo da bisseção,
# x0 de Newton e valores da secante
funcoes = [
    ("f1(x) = x² - 2",       f1, df1, (1, 2),   1.5, (1, 2)),
    ("f2(x) = x³ - x - 2",   f2, df2, (1, 2),   1.5, (1, 2)),
    ("f3(x) = e^(-x) - x",   f3, df3, (0, 1),   0.5, (0, 1)),
    ("f4(x) = x³ - 2x + 2",  f4, df4, (-2, -1), -2,  (-2, -1))
]

resultados = []

for nome, f, df, intervalo, x0_newton, pontos_secante in funcoes:

    # Bisseção
    raiz, iteracoes, aproximacoes, convergiu = bissecao(
        f, intervalo[0], intervalo[1]
    )

    resultados.append({
        "Função": nome,
        "Método": "Bisseção",
        "Valores iniciais": str(intervalo),
        "Raiz aproximada": raiz,
        "|f(x)| final": abs(f(raiz)) if raiz is not None else None,
        "Iterações/resultado":
            f"{iteracoes} / {'Convergiu' if convergiu else 'Falhou'}"
    })

    # Newton
    raiz, iteracoes, aproximacoes, convergiu = newton(
        f, df, x0_newton
    )

    resultados.append({
        "Função": nome,
        "Método": "Newton",
        "Valores iniciais": str(x0_newton),
        "Raiz aproximada": raiz,
        "|f(x)| final": abs(f(raiz)) if raiz is not None else None,
        "Iterações/resultado":
            f"{iteracoes} / {'Convergiu' if convergiu else 'Falhou'}"
    })

    # Secante
    raiz, iteracoes, aproximacoes, convergiu = secante(
        f, pontos_secante[0], pontos_secante[1]
    )

    resultados.append({
        "Função": nome,
        "Método": "Secante",
        "Valores iniciais": str(pontos_secante),
        "Raiz aproximada": raiz,
        "|f(x)| final": abs(f(raiz)) if raiz is not None else None,
        "Iterações/resultado":
            f"{iteracoes} / {'Convergiu' if convergiu else 'Falhou'}"
    })


# Criação da tabela
tabela = pd.DataFrame(resultados)

# Formatação dos números
tabela["Raiz aproximada"] = tabela["Raiz aproximada"].map(
    lambda x: f"{x:.8f}" if x is not None else "-"
)

tabela["|f(x)| final"] = tabela["|f(x)| final"].map(
    lambda x: f"{x:.2e}" if x is not None else "-"
)

tabela

,Função,Método,Valores iniciais,Raiz aproximada,|f(x)| final,Iterações/resultado
0,f1(x) = x² - 2,Bisseção,"(1, 2)",1.41421413,1.62e-06,20 / Convergiu
1,f1(x) = x² - 2,Newton,1.5,1.41421356,4.51e-12,3 / Convergiu
2,f1(x) = x² - 2,Secante,"(1, 2)",1.41421356,8.93e-10,5 / Convergiu
3,f2(x) = x³ - x - 2,Bisseção,"(1, 2)",1.52138042,4.27e-06,20 / Convergiu
4,f2(x) = x³ - x - 2,Newton,1.5,1.52137981,5.89e-07,2 / Convergiu
5,f2(x) = x³ - x - 2,Secante,"(1, 2)",1.52137971,7.02e-09,6 / Convergiu
6,f3(x) = e^(-x) - x,Bisseção,"(0, 1)",0.56714344,2.35e-07,20 / Convergiu
7,f3(x) = e^(-x) - x,Newton,0.5,0.56714317,1.96e-07,2 / Convergiu
8,f3(x) = e^(-x) - x,Secante,"(0, 1)",0.56714331,2.54e-08,4 / Convergiu
9,f4(x) = x³ - 2x + 2,Bisseção,"(-2, -1)",-1.76929188,3.52e-06,20 / Convergiu


Exercício 3

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import display, Image

# ============================================================
# FUNÇÕES
# ============================================================

def f1(x):
    return x**2 - 2

def f2(x):
    return x**3 - x - 2

def df2(x):
    return 3*x**2 - 1

def f3(x):
    return np.exp(-x) - x


# ============================================================
# 1. ANIMAÇÃO - BISSEÇÃO
# ============================================================

def historico_bissecao(f, a, b, tol=1e-6, max_iter=100):
    historico = []

    if f(a) * f(b) > 0:
        raise ValueError("O intervalo não apresenta mudança de sinal.")

    x_anterior = None

    for k in range(1, max_iter + 1):

        x = (a + b) / 2

        historico.append({
            "iteracao": k,
            "a": a,
            "b": b,
            "x": x,
            "fx": f(x)
        })

        if abs(f(x)) < tol:
            break

        if x_anterior is not None and abs(x - x_anterior) < tol:
            break

        if f(a) * f(x) < 0:
            b = x
        else:
            a = x

        x_anterior = x

    return historico


def animar_bissecao(f, a, b, nome_arquivo):

    historico = historico_bissecao(f, a, b)

    # Intervalo usado apenas para desenhar a função
    largura = b - a
    xmin = a - 0.3 * largura
    xmax = b + 0.3 * largura

    xs = np.linspace(xmin, xmax, 500)
    ys = f(xs)

    ymin = min(ys)
    ymax = max(ys)
    margem = 0.15 * (ymax - ymin)

    fig, ax = plt.subplots(figsize=(8, 5))

    def atualizar(frame):

        ax.clear()

        dados = historico[frame]

        a_atual = dados["a"]
        b_atual = dados["b"]
        x = dados["x"]
        fx = dados["fx"]
        k = dados["iteracao"]

        # Função
        ax.plot(xs, ys, label="f(x)")

        # Eixo horizontal
        ax.axhline(0, linewidth=1)

        # Intervalo atual
        ax.axvspan(
            a_atual,
            b_atual,
            alpha=0.2,
            label="Intervalo atual"
        )

        ax.axvline(a_atual, linestyle="--")
        ax.axvline(b_atual, linestyle="--")

        # Aproximação atual
        ax.scatter(x, fx, s=80, zorder=5)
        ax.scatter(x, 0, s=50, zorder=5)

        # Linha entre o ponto da função e o eixo x
        ax.plot(
            [x, x],
            [0, fx],
            linestyle=":"
        )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin - margem, ymax + margem)

        ax.set_xlabel("x")
        ax.set_ylabel("f(x)")
        ax.grid(alpha=0.3)

        ax.set_title(
            f"Bisseção - Iteração {k}\n"
            f"x = {x:.8f}    |f(x)| = {abs(fx):.2e}"
        )

        ax.legend(loc="upper left")

    animacao = FuncAnimation(
        fig,
        atualizar,
        frames=len(historico),
        interval=800,
        repeat=False
    )

    animacao.save(
        nome_arquivo,
        writer=PillowWriter(fps=1)
    )

    plt.close(fig)

    print(f"GIF salvo: {nome_arquivo}")


# ============================================================
# 2. ANIMAÇÃO - NEWTON
# ============================================================

def historico_newton(f, df, x0, tol=1e-6, max_iter=100):

    historico = []
    x = x0

    for k in range(1, max_iter + 1):

        fx = f(x)
        dfx = df(x)

        if abs(dfx) < 1e-12:
            break

        x_novo = x - fx / dfx

        historico.append({
            "iteracao": k,
            "x": x,
            "fx": fx,
            "dfx": dfx,
            "x_novo": x_novo,
            "fx_novo": f(x_novo)
        })

        if abs(x_novo - x) < tol or abs(f(x_novo)) < tol:
            break

        x = x_novo

    return historico


def animar_newton(f, df, x0, nome_arquivo):

    historico = historico_newton(f, df, x0)

    aproximacoes = [d["x"] for d in historico]
    aproximacoes += [d["x_novo"] for d in historico]

    xmin = min(aproximacoes) - 0.5
    xmax = max(aproximacoes) + 0.5

    xs = np.linspace(xmin, xmax, 500)
    ys = f(xs)

    ymin = min(ys)
    ymax = max(ys)

    margem = 0.15 * (ymax - ymin)

    fig, ax = plt.subplots(figsize=(8, 5))

    def atualizar(frame):

        ax.clear()

        dados = historico[frame]

        x = dados["x"]
        fx = dados["fx"]
        dfx = dados["dfx"]

        x_novo = dados["x_novo"]
        fx_novo = dados["fx_novo"]

        k = dados["iteracao"]

        # Função
        ax.plot(xs, ys, label="f(x)")

        # Eixo horizontal
        ax.axhline(0, linewidth=1)

        # Reta tangente
        tangente = fx + dfx * (xs - x)

        ax.plot(
            xs,
            tangente,
            linestyle="--",
            label="Reta tangente"
        )

        # Ponto onde a tangente toca a função
        ax.scatter(x, fx, s=80, zorder=5)

        # Nova aproximação no eixo x
        ax.scatter(
            x_novo,
            0,
            s=70,
            zorder=5,
            label="Nova aproximação"
        )

        # Ponto correspondente na função
        ax.scatter(
            x_novo,
            fx_novo,
            s=50,
            zorder=5
        )

        ax.plot(
            [x_novo, x_novo],
            [0, fx_novo],
            linestyle=":"
        )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin - margem, ymax + margem)

        ax.set_xlabel("x")
        ax.set_ylabel("f(x)")
        ax.grid(alpha=0.3)

        ax.set_title(
            f"Newton - Iteração {k}\n"
            f"x = {x_novo:.8f}    "
            f"|f(x)| = {abs(fx_novo):.2e}"
        )

        ax.legend(loc="upper left")

    animacao = FuncAnimation(
        fig,
        atualizar,
        frames=len(historico),
        interval=1000,
        repeat=False
    )

    animacao.save(
        nome_arquivo,
        writer=PillowWriter(fps=1)
    )

    plt.close(fig)

    print(f"GIF salvo: {nome_arquivo}")


# ============================================================
# 3. ANIMAÇÃO - SECANTE
# ============================================================

def historico_secante(f, x0, x1, tol=1e-6, max_iter=100):

    historico = []

    for k in range(1, max_iter + 1):

        f0 = f(x0)
        f1_atual = f(x1)

        denominador = f1_atual - f0

        if abs(denominador) < 1e-12:
            break

        x2 = x1 - f1_atual * (x1 - x0) / denominador
        f2 = f(x2)

        historico.append({
            "iteracao": k,
            "x0": x0,
            "x1": x1,
            "x2": x2,
            "f0": f0,
            "f1": f1_atual,
            "f2": f2
        })

        if abs(x2 - x1) < tol or abs(f2) < tol:
            break

        x0 = x1
        x1 = x2

    return historico


def animar_secante(f, x0, x1, nome_arquivo):

    historico = historico_secante(f, x0, x1)

    aproximacoes = []

    for dados in historico:
        aproximacoes.extend([
            dados["x0"],
            dados["x1"],
            dados["x2"]
        ])

    xmin = min(aproximacoes) - 0.5
    xmax = max(aproximacoes) + 0.5

    xs = np.linspace(xmin, xmax, 500)
    ys = f(xs)

    ymin = min(ys)
    ymax = max(ys)

    margem = 0.15 * (ymax - ymin)

    fig, ax = plt.subplots(figsize=(8, 5))

    def atualizar(frame):

        ax.clear()

        dados = historico[frame]

        xa = dados["x0"]
        xb = dados["x1"]
        xc = dados["x2"]

        fa = dados["f0"]
        fb = dados["f1"]
        fc = dados["f2"]

        k = dados["iteracao"]

        # Função
        ax.plot(xs, ys, label="f(x)")

        # Eixo horizontal
        ax.axhline(0, linewidth=1)

        # Inclinação da reta secante
        m = (fb - fa) / (xb - xa)

        reta_secante = fa + m * (xs - xa)

        ax.plot(
            xs,
            reta_secante,
            linestyle="--",
            label="Reta secante"
        )

        # Dois pontos usados para construir a secante
        ax.scatter(
            [xa, xb],
            [fa, fb],
            s=70,
            zorder=5,
            label="Pontos da secante"
        )

        # Nova aproximação
        ax.scatter(
            xc,
            0,
            s=70,
            zorder=5,
            label="Nova aproximação"
        )

        # Ponto na função
        ax.scatter(
            xc,
            fc,
            s=50,
            zorder=5
        )

        ax.plot(
            [xc, xc],
            [0, fc],
            linestyle=":"
        )

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin - margem, ymax + margem)

        ax.set_xlabel("x")
        ax.set_ylabel("f(x)")
        ax.grid(alpha=0.3)

        ax.set_title(
            f"Secante - Iteração {k}\n"
            f"x = {xc:.8f}    "
            f"|f(x)| = {abs(fc):.2e}"
        )

        ax.legend(loc="upper left")

    animacao = FuncAnimation(
        fig,
        atualizar,
        frames=len(historico),
        interval=1000,
        repeat=False
    )

    animacao.save(
        nome_arquivo,
        writer=PillowWriter(fps=1)
    )

    plt.close(fig)

    print(f"GIF salvo: {nome_arquivo}")


# ============================================================
# CRIAÇÃO DAS TRÊS ANIMAÇÕES
# ============================================================

animar_bissecao(
    f1,
    1,
    2,
    "bissecao_f1.gif"
)

animar_newton(
    f2,
    df2,
    1.5,
    "newton_f2.gif"
)

animar_secante(
    f3,
    0,
    1,
    "secante_f3.gif"
)

GIF salvo: bissecao_f1.gif
GIF salvo: newton_f2.gif
GIF salvo: secante_f3.gif
